In [117]:
import pandas as pd
import os
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.metrics import brier_score_loss

In [118]:
'''
Function to merge two dataframes
param df1: first dataframe
param df2: second dataframe
return: a single dataframe of the merged dataframes 
'''
def mergeDataframes(df1, df2):
    df = pd.concat([df1, df2])
    return df


'''
Function to merge tournament data for mens and womens and crop the data to match reg detail
params df1: dataframe of mens tournament data
params df2: dataframe of womens tournament data
return: dataframe of combined and cropped tournament data
'''
def mergeTournamentData(df1, df2):
    df1 = df1[df1['Season'] >= 2003].reset_index(drop=True)
    df2 = df2[df2['Season'] >= 2010].reset_index(drop=True)
    
    df = pd.concat([df1, df2])
    return df


'''
Function to split regular season detailed results into dataframes focused on outcome for one team
param df: regular season data
return: a dataframe where each team from a single row in reg data has its own row
'''
def regularDetailsFocus(df):
    RegWinners = pd.DataFrame()
    RegLossers = pd.DataFrame()

    # Establish new columns for that includes stats for one team
    columns = ['Season', 'TeamID', 'Score', 'OppScore',
               'NumOT', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
               'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF', 'OppFGM', 'OppFGA',
               'OppFGM3', 'OppFGA3', 'OppFTM', 'OppFTA', 'OppOR', 'OppDR', 'OppAst', 'OppTO',
               'OppStl', 'OppBlk', 'OppPF']

    # Split winners from regular season
    RegWinners[columns] = df[['Season', 'WTeamID', 'WScore', 'LScore',
                              'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA',
                              'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA',
                              'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO',
                              'LStl', 'LBlk', 'LPF']]

    # Add wins and losses columns
    RegWinners['Wins'] = 1
    RegWinners['Losses'] = 0

    # Split lossers from regular season
    RegLossers[columns] = df[['Season', 'LTeamID', 'LScore', 'WScore',
                               'NumOT', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA',
                               'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF', 'WFGM', 'WFGA',
                               'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO',
                               'WStl', 'WBlk', 'WPF']]

    # Add wins and losses columns
    RegLossers['Wins'] = 0
    RegLossers['Losses'] = 1

    # Combine all games into one dataframe
    AllRegDetail = pd.concat([RegWinners, RegLossers])
    return AllRegDetail


'''
Function to clean seed column
param seeds: Dataframe of historical seeds with column 'Seed'
return: A dataframe with the 'Seed' column converted to int
'''

def cleanSeed(seeds):
    seeds['Seed'] = seeds['Seed'].str.extract(r'(\d+)').astype(int)
    return seeds


'''
Function to join two Dataframes on 'TeamID'
param seeds: Dataframe of historical seeds with column 'Seed'
param tourny: Dataframe of compact tournament data with columns 'WTeamID' and 'LTeamID'
return: a single dataframe of the joined Dataframes
'''

def joinSeeds(seeds, tourny):
    seeds = seeds.set_index(['Season','TeamID'])

    tourny = tourny.join(
        seeds.rename(columns={'Seed':'WSeed'}),
        on=['Season','WTeamID']
    )

    tourny = tourny.join(
        seeds.rename(columns={'Seed':'LSeed'}),
        on=['Season','LTeamID']
    )
    return tourny

In [119]:
# Mens data import
mRegDetail = pd.read_csv('data/men/MRegularSeasonDetailedResults.csv')
mTournCompact = pd.read_csv('data/men/MNCAATourneyCompactResults.csv')
mTournSeeds = pd.read_csv('data/men/MNCAATourneySeeds.csv')
mNames = pd.read_csv('data/men/MTeamSpellings.csv')

# Womens data import
wRegDetail = pd.read_csv('data/women/WRegularSeasonDetailedResults.csv')
wTournCompact = pd.read_csv('data/women/WNCAATourneyCompactResults.csv')
wTournSeeds = pd.read_csv('data/women/WNCAATourneySeeds.csv')
wNames = pd.read_csv('data/women/WTeamSpellings.csv')

# Clean and merge seeds with tournament results
mCleanSeeds = cleanSeed(mTournSeeds.copy())
wCleanSeeds = cleanSeed(wTournSeeds.copy())
mFullTourn = joinSeeds(mCleanSeeds, mTournCompact)
wFullTourn = joinSeeds(wCleanSeeds, wTournCompact)

# Combined data
regDetail = mergeDataframes(mRegDetail, wRegDetail)
compactTourn = mergeTournamentData(mFullTourn, wFullTourn)
names = mergeDataframes(mNames, wNames)

# Split regular season detailed results into dataframes focused on outcome for one team
AllRegDetail = regularDetailsFocus(regDetail)